In [2]:
# Install required libraries
!pip install -q transformers torch

In [3]:
# Q3 - ESG Message Triage using Hugging Face Zero-Shot Classification
# Model: facebook/bart-large-mnli (Zero-shot classifier)

from transformers import pipeline

# Load zero-shot classification model
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# ESG messages to classify
messages = [
    "There is a water leak in Building C that has been running all morning.",
    "The recycling bins are contaminated again and no one seems to be checking them.",
    "The air conditioning is running overnight in an empty office.",
    "I want to report that one of our suppliers may not meet our sustainability policy.",
    "The accessible entrance near the main building has been blocked for two days."
]

# Candidate labels for classification
candidate_labels = ["energy waste", "water leak", "recycling issue", "supplier compliance", "accessibility concern"]

# Classify each message
for message in messages:
    result = classifier(message, candidate_labels)
    print(f"Message: {message}")
    print(f"Top category: {result['labels'][0]}")
    print(f"Confidence: {result['scores'][0]:.4f}")
    print("-" * 60)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Message: There is a water leak in Building C that has been running all morning.
Top category: water leak
Confidence: 0.9564
------------------------------------------------------------
Message: The recycling bins are contaminated again and no one seems to be checking them.
Top category: recycling issue
Confidence: 0.9007
------------------------------------------------------------
Message: The air conditioning is running overnight in an empty office.
Top category: energy waste
Confidence: 0.4462
------------------------------------------------------------
Message: I want to report that one of our suppliers may not meet our sustainability policy.
Top category: supplier compliance
Confidence: 0.5303
------------------------------------------------------------
Message: The accessible entrance near the main building has been blocked for two days.
Top category: accessibility concern
Confidence: 0.9838
------------------------------------------------------------


In [4]:
# Improved ESG classification with JSON output including urgency and sentiment
import json

messages = [
    "There is a water leak in Building C that has been running all morning.",
    "The recycling bins are contaminated again and no one seems to be checking them.",
    "The air conditioning is running overnight in an empty office.",
    "I want to report that one of our suppliers may not meet our sustainability policy.",
    "The accessible entrance near the main building has been blocked for two days."
]

def get_team(category):
    teams = {
        "energy waste": "Facilities Management",
        "water leak": "Facilities Management",
        "recycling issue": "Sustainability Team",
        "supplier compliance": "Procurement Team",
        "accessibility concern": "HR and Compliance"
    }
    return teams.get(category, "General Operations")

def classify_esg_message(message, classifier):
    issue_categories = ["energy waste", "water leak", "recycling issue",
                       "supplier compliance", "accessibility concern"]
    urgency_levels = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
    sentiment_labels = ["POSITIVE", "NEUTRAL", "NEGATIVE"]

    category_result = classifier(message, issue_categories)
    urgency_result = classifier(message, urgency_levels)
    sentiment_result = classifier(message, sentiment_labels)
    followup = "Y" if urgency_result['labels'][0] in ["HIGH", "CRITICAL"] else "N"

    output = {
        "message": message,
        "issue_category": category_result['labels'][0],
        "urgency": urgency_result['labels'][0],
        "sentiment": sentiment_result['labels'][0],
        "followup_required": followup,
        "recommended_team": get_team(category_result['labels'][0]),
        "confidence_score": round(category_result['scores'][0], 4)
    }
    return output

print("=== ESG MESSAGE TRIAGE - DETAILED JSON OUTPUT ===\n")
for message in messages:
    result = classify_esg_message(message, classifier)
    print(json.dumps(result, indent=2))
    print("-" * 60)

=== ESG MESSAGE TRIAGE - DETAILED JSON OUTPUT ===

{
  "message": "There is a water leak in Building C that has been running all morning.",
  "issue_category": "water leak",
  "urgency": "MEDIUM",
  "sentiment": "NEGATIVE",
  "followup_required": "N",
  "recommended_team": "Facilities Management",
  "confidence_score": 0.9564
}
------------------------------------------------------------
{
  "message": "The recycling bins are contaminated again and no one seems to be checking them.",
  "issue_category": "recycling issue",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Sustainability Team",
  "confidence_score": 0.9007
}
------------------------------------------------------------
{
  "message": "The air conditioning is running overnight in an empty office.",
  "issue_category": "energy waste",
  "urgency": "MEDIUM",
  "sentiment": "NEUTRAL",
  "followup_required": "N",
  "recommended_team": "Facilities Management",
  "confidence_score

In [5]:
# Comparison: Basic Zero-Shot vs Improved JSON Classification
print("=== COMPARISON: BASIC vs IMPROVED CLASSIFICATION ===\n")
print(f"{'Message':<50} {'Basic Category':<20} {'Improved Urgency':<20} {'Followup'}")
print("-" * 100)

basic_results = [
    ("Water leak in Building C", "water leak", "MEDIUM", "N"),
    ("Recycling bins contaminated", "recycling issue", "HIGH", "Y"),
    ("AC running overnight", "energy waste", "MEDIUM", "N"),
    ("Supplier sustainability issue", "supplier compliance", "LOW", "N"),
    ("Accessible entrance blocked", "accessibility concern", "LOW", "N")
]

for msg, category, urgency, followup in basic_results:
    print(f"{msg:<50} {category:<20} {urgency:<20} {followup}")

print("\nKey observations:")
print("1. Basic classifier correctly identifies categories with high confidence")
print("2. Improved classifier adds urgency, sentiment and routing information")
print("3. Water leak flagged as MEDIUM urgency - could argue should be HIGH")
print("4. Low confidence (0.44) for AC issue suggests model uncertainty")
print("5. Accessibility concern correctly routed to HR and Compliance team")

=== COMPARISON: BASIC vs IMPROVED CLASSIFICATION ===

Message                                            Basic Category       Improved Urgency     Followup
----------------------------------------------------------------------------------------------------
Water leak in Building C                           water leak           MEDIUM               N
Recycling bins contaminated                        recycling issue      HIGH                 Y
AC running overnight                               energy waste         MEDIUM               N
Supplier sustainability issue                      supplier compliance  LOW                  N
Accessible entrance blocked                        accessibility concern LOW                  N

Key observations:
1. Basic classifier correctly identifies categories with high confidence
2. Improved classifier adds urgency, sentiment and routing information
3. Water leak flagged as MEDIUM urgency - could argue should be HIGH
4. Low confidence (0.44) for AC iss